# SFT LoRA — Qwen3.5-2B

Chấm bằng **đúng code của BTC** (`third_party/scoring.py` + `rouge_score/`), chọn
checkpoint theo **METEOR**, không theo loss và không theo trung bình hai metric. Lý do đo
được, trên 300 cặp thật:

| baseline | METEOR | ROUGE-L |
|---|---:|---:|
| chính reference | 1,0000 | 1,0000 |
| nửa đầu reference | 0,4791 | 0,6650 |
| reference xáo từ | 0,5080 | 0,3524 |
| reference của câu KHÁC | 0,1170 | 0,2780 |

ROUGE-L có **sàn 0,278** cho câu sai hẳn (tokenizer của nó xoá dấu, khớp mảnh ASCII) và
**thưởng trả lời cụt** (0,665 cho nửa đầu). Chọn theo trung bình là dạy model nói ngắn.

Loss cũng không dùng làm tiêu chí dừng: ~88% đáp án mở đầu cùng khuôn "Căn cứ Điều N…",
loss vẫn giảm đúng lúc model chuyển sang học thuộc khuôn — đúng dạng lỗi của v9.2.

**Ngân sách.** T4 ~3 s/mẫu (seq p50 3.378, LoRA + gradient checkpointing) → ~4,5 giờ cho
một epoch 5.400 mẫu, cộng ~15 phút mỗi lần eval sinh 120 câu × 768 token. Phiên Kaggle
chết ở 12 giờ và **output không được lưu nếu kernel chết vì hết giờ**, nên
`--gio-toi-da 10.5` dừng chủ động và ghi `cuoi/`. Phiên sau chạy tiếp bằng `--tiep`.

In [ ]:
import os, sys, shutil, zipfile, json
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent
        break
if SRC is None:
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

!pip -q install transformers==4.44.2 peft==0.13.2 sentencepiece nltk

os.chdir(REPO)
sys.path.insert(0, "src")
os.environ["PYTHONPATH"] = "src"

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())
print("GPU     :", torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability()
print("sm      :", f"{cap[0]}.{cap[1]}", "(fp16 OK)" if cap[0] >= 7 else "(fp16 KHÔNG chạy — đổi sang T4)")

## Dữ liệu và model nền

`sft.jsonl` lấy từ output kernel `v10-build-sft`, `dev.jsonl` từ `v10-build-dev`. Cả hai
vào bằng *Add Input → Notebook Output*, không dựng lại — dựng lại tốn 3,5 giờ GPU.

In [ ]:
SFT = next(Path("/kaggle/input").glob("**/sft.jsonl"), None)
DEV = next(Path("/kaggle/input").glob("**/dev.jsonl"), None)
assert SFT and DEV, f"thiếu dữ liệu: sft={SFT} dev={DEV} — gắn output v10-build-sft và v10-build-dev"

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/config.json")
             if "Qwen" in p.read_text() and (p.parent / "tokenizer.json").exists()
             and (p.parent / "config.json").stat().st_size > 0
             and "ForCausalLM" in p.read_text()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"

n_sft = sum(1 for _ in open(SFT, encoding="utf-8"))
n_dev = sum(1 for _ in open(DEV, encoding="utf-8"))
print(f"train {n_sft:,}  dev {n_dev:,}")
print("base  ", BASE)
os.environ["SFT"], os.environ["DEV"], os.environ["BASE"] = str(SFT), str(DEV), str(BASE)

## Train

`--eval-steps 100` với `accum 16` là eval mỗi 1.600 mẫu — đủ dày để thấy đường METEOR
uốn, đủ thưa để phần sinh không nuốt mất giờ GPU. `--kien-nhan 3` dừng sớm khi ba lần
eval liên tiếp không cải thiện METEOR.

Đọc log theo ba số: `loss` giảm đều, `gnorm` **không** vọt (ghi trước khi clip nên nó
nói thật), và `METEOR` ở mỗi mốc. Nếu `meteor_truot` bám sát `meteor` thì model đang bịa
tự tin khi context thiếu căn cứ — tập train đã loại hết nhóm đó nên nó chưa từng thấy.

In [ ]:
!python scripts/train_sft.py --model $BASE --data $SFT --dev $DEV \
    --ra /kaggle/working/out --epochs 1.0 --lr 1e-4 --r 32 --alpha 64 \
    --accum 16 --eval-steps 100 --dev-n 120 --max-new 768 --gen-bs 4 \
    --gio-toi-da 10.5 2>&1 | tail -400

In [ ]:
h = json.loads(Path("/kaggle/working/out/lich_su.json").read_text())
print(f"{'bước':>6} {'loss':>7} {'gnorm':>6} {'METEOR':>7} {'ROUGE':>7} {'M-trượt':>8} {'dài':>6}")
for d in h:
    print(f"{d['buoc']:6} {d['loss']:7.4f} {d['gnorm']:6.2f} {d['meteor']:7.4f} "
          f"{d['rouge']:7.4f} {d.get('meteor_truot', float('nan')):8.4f} {d['do_dai_tb']:6.0f}")